<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 8 (backbone screen with frozen features)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 15 to 20 min).

Run 7 showed that averaging the three YOLOv8s-cls seeds raised the cross-validated ROC AUC from 0.894 to 0.912, the only change so far that improved ranking. Averaging *different* architectures usually helps more, because they make different mistakes. Fine-tuning every candidate would take hours, so Run 8 screens them cheaply first:

1. Each pretrained backbone is **frozen** and turns every photo into a feature vector, averaged over four flipped copies of the photo (none, horizontal, vertical, both).
2. A logistic regression is scored on those features with the **same 5 folds** as Runs 6 and 7 (the regularization strength is chosen inside each training fold).
3. The table ranks backbones by out-of-fold ROC AUC, and a second table scores **pairs** of backbones averaged together, which shows which ones add something to YOLOv8s-cls.

For reference, frozen YOLOv8s-cls features scored about 0.857 in Run 7's label-cleaning step, and the fine-tuned Run 7 models scored 0.894 (single) and 0.912 (3-seed average). A frozen screen understates every backbone; what matters is the ranking and which pairs complement each other. The **test set is not loaded or scored**. The winners go to Run 9 for fine-tuning.

Sizes are for weights saved in half precision and still include the 1000-class ImageNet head, so they run slightly above the fine-tuned size (YOLOv8s-cls: about 12.7 MB here vs. 10.3 MB fine-tuned). CPU times are forward passes on Colab's CPU and only a rough guide, since the success criteria measure latency on a team laptop.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub

In [ ]:
import os, glob, shutil, time, json, random, math, itertools
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch, torchvision
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, recall_score, roc_auc_score

SEED = 42
FOLDS = 5
RECALL_MIN = 0.90
MAX_MB = 50
NEAR_DUP_T = 5
DATA_SRC = None
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"
TV_WEIGHTS = True            # torchvision ImageNet weights (set False only for an offline test)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME)
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
OUT = os.path.join(BASE, "results", "run8"); os.makedirs(OUT, exist_ok=True)
T_START = time.time()

## 1. Data: the cleaned split and the Run 6 folds

Same code as Runs 6 and 7. Only the 761 training + validation photos are used; the test photos are set aside and never loaded.

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)

splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
assert int(df["split"].notna().sum()) == len(splits1), "splits.csv did not match the download"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = set(n_lab[n_lab > 1].index)
clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
rank = {"train": 0, "val": 1, "test": 2}
drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in zip(ii, jj) if sp[i] != sp[j] and lb[i] == lb[j]}
drop |= {k for i, j in zip(ii, jj) if lb[i] != lb[j] for k in (i, j)}
clean = clean.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Removed {len(conflict_hashes)} conflicting-label groups and {len(drop)} near-duplicates")
print(pd.crosstab(clean["split"], clean["label"], margins=True))

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

POOL = os.path.join(BASE, "coral_pool"); shutil.rmtree(POOL, ignore_errors=True)
imgs = []
for _, r in clean.iterrows():
    d = os.path.join(POOL, r["label"]); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg")
    pad_square(r["path"]).save(dst, quality=95); imgs.append(dst)
clean["img"] = imgs
assert clean["img"].nunique() == len(clean), "file name clash in the image pool"

trval = clean[clean["split"] != "test"].reset_index(drop=True)
test = clean[clean["split"] == "test"].reset_index(drop=True)
y_trval = (trval["label"] == "bleached").astype(int).to_numpy()
y_test = (test["label"] == "bleached").astype(int).to_numpy()
test_files = test["img"].tolist()
FOLD_IDX = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(trval)), y_trval))
fold_of = np.zeros(len(trval), dtype=int)
for k, (_, va) in enumerate(FOLD_IDX): fold_of[va] = k
trval.assign(fold=fold_of)[["file", "label", "split", "fold"]].to_csv(os.path.join(OUT, "folds.csv"), index=False)
test[["file", "label"]].to_csv(os.path.join(OUT, "test_images.csv"), index=False)
print(f"Train + validation: {len(trval)} ({y_trval.sum()} bleached) in {FOLDS} folds | test: {len(test)} ({y_test.sum()} bleached)")

## 2. Backbones

| Backbone | Pretraining | Input | Why it is here |
| :-- | :-- | :-- | :-- |
| YOLOv8s-cls | ImageNet | 224 | the current model |
| YOLOv8m-cls | ImageNet | 224 | larger YOLO |
| EfficientNet-B0 | ImageNet | 224 | the comparison model from the proposal |
| EfficientNet-B2 | ImageNet | 288 | larger EfficientNet |
| EfficientNetV2-S | ImageNet | 384 | newer training recipe |
| RegNetY-1.6GF | ImageNet | 224 | a different CNN design |
| DINOv2 ViT-S/14 | self-supervised, 142M images | 224 | a transformer with very general features |

In [ ]:
IMNET_MEAN, IMNET_STD = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

class YoloFeatures(nn.Module):
    """Pooled 1280-d features of an Ultralytics classifier (the input of its last linear layer)."""
    def __init__(self, weights):
        super().__init__()
        from ultralytics import YOLO
        self.net = YOLO(weights).model.float().eval(); self._f = None
        self.net.model[-1].linear.register_forward_hook(lambda m, i, o: setattr(self, "_f", i[0]))
    def forward(self, x):
        self.net(x); return self._f

class TvFeatures(nn.Module):
    """torchvision classifier with its last layer removed."""
    def __init__(self, name, weights):
        super().__init__()
        self.net = getattr(torchvision.models, name)(weights=weights if TV_WEIGHTS else None)
        if hasattr(self.net, "classifier"): self.net.classifier = nn.Identity()
        else: self.net.fc = nn.Identity()
    def forward(self, x):
        return self.net((x - IMNET_MEAN.to(x.device)[:, None, None]) / IMNET_STD.to(x.device)[:, None, None])

def load_dino_backbone():
    return torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)

class DinoFeatures(nn.Module):
    def __init__(self):
        super().__init__(); self.net = load_dino_backbone()
    def forward(self, x):
        f = self.net.forward_features((x - IMNET_MEAN.to(x.device)[:, None, None]) / IMNET_STD.to(x.device)[:, None, None])
        return torch.cat([f["x_norm_clstoken"], f["x_norm_patchtokens"].mean(1)], 1)

BACKBONES = {
    "YOLOv8s-cls":     (lambda: YoloFeatures("yolov8s-cls.pt"), 224),
    "YOLOv8m-cls":     (lambda: YoloFeatures("yolov8m-cls.pt"), 224),
    "EfficientNet-B0": (lambda: TvFeatures("efficientnet_b0", "IMAGENET1K_V1"), 224),
    "EfficientNet-B2": (lambda: TvFeatures("efficientnet_b2", "IMAGENET1K_V1"), 288),
    "EfficientNetV2-S": (lambda: TvFeatures("efficientnet_v2_s", "IMAGENET1K_V1"), 384),
    "RegNetY-1.6GF":   (lambda: TvFeatures("regnet_y_1_6gf", "IMAGENET1K_V2"), 224),
    "DINOv2 ViT-S/14": (lambda: DinoFeatures(), 224),
}

def load_tensor(files, size):
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1) for p in files])

def pick_threshold(y, p):
    rows = [(t, f1_score(y, p >= t, average="macro"), recall_score(y, p >= t, zero_division=0)) for t in np.round(np.linspace(0.01, 0.99, 981), 3)]
    ok = [r for r in rows if r[2] >= RECALL_MIN]
    return max(ok or rows, key=lambda r: (r[1], r[0]))

## 3. Features, size, and CPU time for each backbone

In [ ]:
FEATS, info = {}, []
files = trval["img"].tolist()
for name, (build, size) in BACKBONES.items():
    t0 = time.time()
    try:
        net = build().eval().to(TDEV)
    except Exception as e:
        print(f"{name}: skipped ({e!r})"); continue
    X = load_tensor(files, size); out = []
    with torch.no_grad():
        for k in range(0, len(X), 32):
            xb = X[k:k + 32].to(TDEV).float() / 255
            out.append(torch.stack([net(v).float() for v in (xb, xb.flip(3), xb.flip(2), xb.flip(2).flip(3))]).mean(0).cpu())
    FEATS[name] = torch.cat(out).numpy()
    params = sum(p.numel() for p in net.parameters())
    cpu = net.cpu(); x1 = torch.rand(1, 3, size, size)
    with torch.no_grad():
        for _ in range(3): cpu(x1)
        t1 = time.perf_counter()
        for _ in range(10): cpu(x1)
    ms = (time.perf_counter() - t1) / 10 * 1000
    info.append(dict(backbone=name, input=size, params_m=round(params / 1e6, 1), size_mb_fp16=round(params * 2 / 1e6, 1),
                     colab_cpu_forward_ms=round(ms), feature_dim=FEATS[name].shape[1]))
    print(f"{name}: {FEATS[name].shape[1]}-d features, {params / 1e6:.1f}M params, {ms:.0f} ms CPU forward ({(time.time() - t0) / 60:.1f} min)")
    del net, cpu, X; torch.cuda.empty_cache()
info = pd.DataFrame(info)

## 4. Out-of-fold scores: single backbones and pairs

Logistic regression with standardized features; its regularization strength is picked by an inner 5-fold search inside each training fold, so the held-out fold never influences it.

In [ ]:
def oof_probe(Xf):
    p = np.zeros(len(y_trval))
    for tr, va in FOLD_IDX:
        clf = make_pipeline(StandardScaler(), LogisticRegressionCV(Cs=[0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0], cv=5, scoring="roc_auc",
                                                                   max_iter=5000, random_state=SEED))
        clf.fit(Xf[tr], y_trval[tr]); p[va] = clf.predict_proba(Xf[va])[:, 1]
    return p

OOF = {}
for name, Xf in FEATS.items():
    t0 = time.time(); OOF[name] = oof_probe(Xf); print(f"{name}: OOF ROC AUC {roc_auc_score(y_trval, OOF[name]):.3f} ({time.time() - t0:.0f} s)")

def row(label, p, mb, ms):
    thr, f1, rec = pick_threshold(y_trval, p)
    return dict(model=label, oof_roc_auc=roc_auc_score(y_trval, p), macro_f1_at_0_5=f1_score(y_trval, p >= 0.5, average="macro"),
                cutoff=thr, macro_f1_at_cutoff=f1, recall_at_cutoff=rec, size_mb_fp16=mb, colab_cpu_forward_ms=ms, within_50mb=mb < MAX_MB)

meta = info.set_index("backbone")
single = pd.DataFrame([row(n, p, meta.loc[n, "size_mb_fp16"], meta.loc[n, "colab_cpu_forward_ms"]) for n, p in OOF.items()]) \
           .sort_values("oof_roc_auc", ascending=False)
pairs = pd.DataFrame([row(f"{a} + {b}", (OOF[a] + OOF[b]) / 2, meta.loc[a, "size_mb_fp16"] + meta.loc[b, "size_mb_fp16"],
                          meta.loc[a, "colab_cpu_forward_ms"] + meta.loc[b, "colab_cpu_forward_ms"]) | {"corr": np.corrcoef(OOF[a], OOF[b])[0, 1]}
                      for a, b in itertools.combinations(OOF, 2)]).sort_values("oof_roc_auc", ascending=False)
single.to_csv(os.path.join(OUT, "screen_single.csv"), index=False); pairs.to_csv(os.path.join(OUT, "screen_pairs.csv"), index=False)
pd.DataFrame({"file": trval["file"], "true_label": y_trval, **OOF}).to_csv(os.path.join(OUT, "oof_frozen_probes.csv"), index=False)
info.to_csv(os.path.join(OUT, "backbones.csv"), index=False)
print("Single backbones (frozen):"); print(single.round(3).to_string(index=False))
print("\nPairs averaged (frozen), top 10:"); print(pairs.head(10).round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.6))
s = single.sort_values("oof_roc_auc")
ax.barh(s["model"], s["oof_roc_auc"], color=["#2a78d6" if w else "#bbbbbb" for w in s["within_50mb"]], height=0.6)
for i, v in enumerate(s["oof_roc_auc"]): ax.text(v + 0.002, i, f"{v:.3f}", va="center", fontsize=8)
ax.axvline(0.894, color="#eb6834", ls="--", lw=1.5); ax.text(0.8945, len(s) - 0.4, "fine-tuned YOLOv8s (Run 7)", fontsize=7, color="#eb6834")
ax.set_xlim(max(0.5, s["oof_roc_auc"].min() - 0.05), max(0.95, s["oof_roc_auc"].max() + 0.02))
ax.set_xlabel("Out-of-fold ROC AUC, frozen features (grey: over 50 MB)"); ax.set_title(f"Run 8 backbone screen, {FOLDS}-fold CV (n={len(trval)})", fontsize=10)
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig(os.path.join(OUT, "screen_single.png"), dpi=150); plt.show()

## 5. Summary and the Run 9 recommendation

Run 9 fine-tunes the best backbone that fits on its own and the best partner for YOLOv8s-cls (highest pair ROC AUC with YOLOv8s-cls whose combined size, with two YOLOv8s-cls seeds, stays under 50 MB).

In [ ]:
fits = single[single["within_50mb"]]
yolo_s = meta.loc["YOLOv8s-cls", "size_mb_fp16"] if "YOLOv8s-cls" in meta.index else 10.3
partners = pairs[pairs["model"].str.contains("YOLOv8s-cls") & ~pairs["model"].str.contains("YOLOv8m")].copy()
partners["partner"] = partners["model"].str.replace("YOLOv8s-cls + ", "").str.replace(" + YOLOv8s-cls", "")
partners["size_with_2_yolo_seeds"] = [2 * yolo_s + meta.loc[p, "size_mb_fp16"] for p in partners["partner"]]
best_partner = partners[partners["size_with_2_yolo_seeds"] < MAX_MB].iloc[0] if len(partners[partners["size_with_2_yolo_seeds"] < MAX_MB]) else None
L = [f"RUN 8 SUMMARY ({time.strftime('%b. %d, %Y')})",
     f"Frozen-feature screen, 4 flips averaged, logistic regression, {FOLDS}-fold CV on train + validation (n={len(trval)}); test set not used.",
     "Single backbones (out-of-fold):"]
L += [f"  {r.model}: ROC AUC {r.oof_roc_auc:.3f}; macro F1 @0.5 {r.macro_f1_at_0_5:.3f}; at cutoff {r.cutoff:.3f}: macro F1 {r.macro_f1_at_cutoff:.3f}, recall {r.recall_at_cutoff:.3f}; "
      f"{r.size_mb_fp16:.1f} MB, {r.colab_cpu_forward_ms:.0f} ms Colab CPU" for r in single.itertuples()]
L += ["Best pairs (average of two frozen probes):"]
L += [f"  {r.model}: ROC AUC {r.oof_roc_auc:.3f}, macro F1 at cutoff {r.macro_f1_at_cutoff:.3f}, correlation {r.corr:.2f}, {r.size_mb_fp16:.1f} MB" for r in pairs.head(6).itertuples()]
L += [f"Best single backbone under 50 MB: {fits.iloc[0]['model']} (ROC AUC {fits.iloc[0]['oof_roc_auc']:.3f}).",
      "Best partner for YOLOv8s-cls within 50 MB with two YOLO seeds: " + (f"{best_partner['partner']} (pair ROC AUC {best_partner['oof_roc_auc']:.3f}, "
      f"{best_partner['size_with_2_yolo_seeds']:.1f} MB total)" if best_partner is not None else "none fits"),
      f"Environment: GPU {GPU_NAME}, torch {torch.__version__}, torchvision {torchvision.__version__}. Total time {(time.time() - T_START) / 60:.1f} min."]
summary = "\n".join(L); print(summary); open(os.path.join(OUT, "run8_summary.txt"), "w").write(summary)
ZIP = shutil.make_archive(os.path.join(BASE, "run8_results"), "zip", OUT); print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass